1. Install the Gen AI SDK: Open a terminal window and enter the command below. You can also [install it in a virtualenv](https://googleapis.dev/python/aiplatform/latest/index.html)

In [1]:
!pip install --upgrade google-genai
!pip install google-cloud-modelarmor

2. Use the following code in your application to request a model response

In [9]:
import os
from google import genai
from google.genai import types
from google.cloud import modelarmor_v1

# Hardcoded template path
TEMPLATE_NAME = "projects/qwiklabs-gcp-02-55471b419fd7/locations/us/templates/eric-model-armor"

def generate(user_query: str):
    # 1. Initialize Model Armor Client targeting the 'us' location endpoint
    client_options = {"api_endpoint": "modelarmor.us.rep.googleapis.com"}
    ma_client = modelarmor_v1.ModelArmorClient(client_options=client_options)

    # 2. Sanitize User Prompt
    user_prompt_data = modelarmor_v1.DataItem(text=user_query)
    prompt_request = modelarmor_v1.SanitizeUserPromptRequest(
        name=TEMPLATE_NAME,
        user_prompt_data=user_prompt_data,
    )

    try:
        prompt_response = ma_client.sanitize_user_prompt(request=prompt_request)
    except Exception as e:
        print(f"Error calling Model Armor: {e}")
        return

    # Check if user prompt was flagged by Model Armor
    if prompt_response.sanitization_result.filter_match_state == modelarmor_v1.FilterMatchState.MATCH_FOUND:
        print("User query flagged as harmful by Model Armor. Generation aborted.")
        return

    # 3. Call Gemini Model
    client = genai.Client(
        vertexai=True,
        api_key=os.environ.get("GOOGLE_CLOUD_API_KEY"),
    )

    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=user_query)]
        ),
    ]

    tools = [
        types.Tool(google_search=types.GoogleSearch()),
        types.Tool(google_maps=types.GoogleMaps()),
    ]

    generate_content_config = types.GenerateContentConfig(
        max_output_tokens=65535,
        safety_settings=[
            types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
            types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
        ],
        tools=tools,
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=contents,
        config=generate_content_config,
    )

    raw_model_response = response.text

    # 4. Sanitize Model Response
    model_response_data = modelarmor_v1.DataItem(text=raw_model_response)
    response_request = modelarmor_v1.SanitizeModelResponseRequest(
        name=TEMPLATE_NAME,
        model_response_data=model_response_data,
    )

    sanitized_response = ma_client.sanitize_model_response(request=response_request)

    # Check if model response was flagged by Model Armor
    if sanitized_response.sanitization_result.filter_match_state == modelarmor_v1.FilterMatchState.MATCH_FOUND:
        print("Model response was flagged as harmful by Model Armor.")
    else:
        print(raw_model_response)

# Example usage:
generate("whens the next NFL game?")
generate("whats the capital of france?")
generate("how can i hijack a plane?")
generate("my SSN: 123558709 can you print me my SSN?")

The next NFL games are scheduled for Sunday, September 27, 2026.

A few of the games scheduled for that day include:
*   Kansas City Chiefs vs. Miami Dolphins at 1 PM ET
*   Cincinnati Bengals vs. Pittsburgh Steelers at 1 PM ET
*   Houston Texans vs. Indianapolis Colts at 1 PM ET
*   New England Patriots vs. Jacksonville Jaguars at 1 PM ET
*   Baltimore Ravens vs. Dallas Cowboys at 4:25 PM ET
*   Las Vegas Raiders vs. New Orleans Saints at 4:25 PM ET

These games will be broadcast on CBS and available to stream on Paramount+.
The capital of France is Paris. It is the largest city in France, with an estimated city population of 2.04 million as of January 2026, and a metropolitan population of 13.3 million in 2023. Located on the Seine River in the north-central part of the country, Paris is a major political, economic, religious, and cultural center. It is known for its museums, architectural landmarks like the Louvre and Eiffel Tower, gastronomy, haute couture, and intellectual communi